
# Cento Sentence-Level Similarity Analysis

Per-sentence LambdaG scores showing which cento sentences are most like Vergil
and least like Ausonius, in both metrical and syntactic representations.

Uses the same models and settings as the stylometry notebooks:
- **Vergil model**: Georgics (4 books) + Background as reference
- **Ausonius model**: Ausonius hexameter works (17 works) + Background as reference
- N=10, r=30, relative scale

For each model we extract the top 20 sentences most similar to the candidate
and display them with their Latin text.


In [1]:

import re
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_corpora import Sentence, TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize


def parse_filename(path: Path, suffix: str) -> tuple[str, str]:
    """Parse '<name><suffix>' into (author, work).

    Filenames follow the pattern:
      - ausonius: ausonius_caesares_sentences.*  →  ('ausonius', 'caesares')
      - vergil: vergil_georgics_1_sentences.*  →  ('vergil', 'georgics_1')
      - aeneid: vergil_aeneid_1_sentences.*  →  ('vergil', 'aeneid_1')
      - background: ovid_metamorphoses_1_sentences.*  →  ('ovid', 'metamorphoses_1')
      - question: cento_nuptialis_sentences.*  →  ('cento', 'nuptialis')
    """
    name = re.sub(re.escape(suffix) + "$", "", path.name)
    parts = name.split("_", 1)
    if len(parts) == 1:
        return parts[0], ""
    return parts[0], parts[1]


def load_corpus(data_dir: Path, suffix: str) -> TwoLevelCorpus:
    """Read all matching sentence files and return a TwoLevelCorpus."""
    sentences = []
    for path in sorted(data_dir.glob(f"*{suffix}")):
        author, work = parse_filename(path, suffix)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(Sentence(text=line, metadata={"author": author, "work": work}))
    return TwoLevelCorpus(sentences, levels=["author", "work"])


def load_plain_sentences(path: Path) -> list[str]:
    """Load sentences as list of raw text strings."""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def to_sentence_lists(corpus: TwoLevelCorpus) -> list[list[str]]:
    """Tokenise corpus sentences into list of token lists."""
    return [s.text.split() for s in corpus]


In [2]:

# Load metrical corpora
m_vergil = load_corpus(Path("metrical_model/vergil"), "_sentences.metrical.txt")
m_ausonius = load_corpus(Path("metrical_model/ausonius"), "_sentences.metrical.txt")
m_background = load_corpus(Path("metrical_model/background"), "_sentences.metrical.txt")
m_cento = load_corpus(Path("metrical_model/question"), "_sentences.metrical.txt")

# Load syntactic corpora
s_vergil = load_corpus(Path("lexical_model/vergil"), "_sentences.quesplit.masked.txt")
s_ausonius = load_corpus(Path("lexical_model/ausonius"), "_sentences.quesplit.masked.txt")
s_background = load_corpus(Path("lexical_model/background"), "_sentences.quesplit.masked.txt")
s_cento = load_corpus(Path("lexical_model/question"), "_sentences.quesplit.masked.txt")

# Load unmasked cento sentences for display
unmasked = load_plain_sentences(Path("processed/question/cento_nuptialis_sentences.txt"))

print("Metrical:")
print(f"  Vergil: {len(m_vergil)} sentences, {sum(len(s.text.split()) for s in m_vergil)} tokens")
print(f"  Ausonius: {len(m_ausonius)} sentences, {sum(len(s.text.split()) for s in m_ausonius)} tokens")
print(f"  Background: {len(m_background)} sentences, {sum(len(s.text.split()) for s in m_background)} tokens")
print(f"  Cento: {len(m_cento)} sentences")
print()
print("Syntactic:")
print(f"  Vergil: {len(s_vergil)} sentences")
print(f"  Ausonius: {len(s_ausonius)} sentences")
print(f"  Background: {len(s_background)} sentences")
print(f"  Cento: {len(s_cento)} sentences")
print()
print(f"Unmasked cento: {len(unmasked)} sentences")


Metrical:
  Vergil: 809 sentences, 17169 tokens
  Ausonius: 959 sentences, 17355 tokens
  Background: 15111 sentences, 295589 tokens
  Cento: 58 sentences

Syntactic:
  Vergil: 809 sentences
  Ausonius: 959 sentences
  Background: 15111 sentences
  Cento: 58 sentences

Unmasked cento: 58 sentences



## LambdaG Visualisation

Run `lambdaG_visualize` on the full cento (all sentences) against each candidate,
then render the raw HTML output. Token colours and circle colours are set by
lambdaG itself — red/warm = positive (similar), blue = negative (dissimilar).

Pipe tokens (`|`) are stripped from the unmasked text so they don't corrupt
the token-position matching between masked and unmasked views.


In [6]:

from IPython.display import IFrame
import re, base64


def strip_pipes(sentences: list[str]) -> list[str]:
    """Remove pipe markers from sentence strings."""
    return [s.replace("|", "").strip() for s in sentences]


def run_lambdag(q_text, unmasked, k_text, ref_text, negative, rng):
    """Run lambdaG_visualize and return the raw result dict."""
    return lambdaG_visualize(
        q_text=q_text,
        k_text=k_text,
        ref_text=ref_text,
        N=10, r=30,
        output="html",
        scale="relative",
        negative=negative,
        order_by="importance",
        unmasked_sentences=unmasked,
        rng=rng,
    )


def render_full(result: dict, title: str, n_sentences: int) -> IFrame:
    """Render the raw lambdaG HTML output (full poem) in an IFrame."""
    html = result["colourcoded_text"]
    html = html.replace("<EOL>", "&#60;EOL&#62;")
    output_html = (
        '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
        '<link href="https://fonts.googleapis.com/css2?family=Roboto+Condensed:wght@300;400;700&display=swap" rel="stylesheet">'
        "<style>"
        "body{font-family:'Roboto Condensed',sans-serif;margin:0;padding:20px;background:#f5f5f5;font-weight:300}"
        "h2{text-align:center;color:#333;font-weight:700}"
        ".subtitle{text-align:center;color:#666;margin-bottom:20px;font-size:14px}"
        "</style></head><body>"
        f"<h2>{title}</h2>"
        f"<p class='subtitle'>{n_sentences} sentences, token-level LambdaG, relative scale, ordered by importance</p>"
        + html + "</body></html>"
    )
    html_bytes = output_html.encode("utf-8")
    html_b64 = base64.b64encode(html_bytes).decode("utf-8")
    return IFrame(f"data:text/html;base64,{html_b64}", width="100%", height=800)


def extract_sentence_rows(html: str) -> list[str]:
    """Extract sentence-row divs from lambdaG HTML output."""
    return re.findall(
        r'<div class="sentence-row">.*?</div>\s*</div>', html, re.DOTALL
    )


def render_rows(rows: list[str], title: str, subtitle: str = "") -> IFrame:
    """Wrap extracted sentence-row divs in a styled HTML page and return IFrame."""
    html = "\n".join(rows).replace("<EOL>", "&#60;EOL&#62;")
    subtitle_html = f"<p class='subtitle'>{subtitle}</p>" if subtitle else ""
    output_html = (
        '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
        '<link href="https://fonts.googleapis.com/css2?family=Roboto+Condensed:wght@300;400;700&display=swap" rel="stylesheet">'
        "<style>"
        "body{font-family:'Roboto Condensed',sans-serif;margin:0;padding:20px;background:#f5f5f5;font-weight:300}"
        "h2{text-align:center;color:#333;font-weight:700}"
        ".subtitle{text-align:center;color:#666;margin-bottom:20px;font-size:14px}"
        ".sentence-row{display:flex;align-items:flex-start;margin-bottom:12px;gap:10px}"
        ".sentence-column{flex:1;line-height:1.6;font-size:15px}"
        ".sentence-number-circle{display:inline-block;width:28px;height:28px;line-height:28px;border-radius:50%;text-align:center;font-size:12px;font-weight:700}"
        ".heading-row{display:flex;align-items:center;margin-bottom:10px;gap:10px}"
        ".heading-spacer{width:38px;flex-shrink:0}"
        ".column-heading{flex:1;text-align:center;color:#555;font-size:1.2em;font-weight:400;border-bottom:2px solid #ddd;padding-bottom:10px}"
        "</style></head><body>"
        f"<h2>{title}</h2>"
        f"{subtitle_html}"
        '<div class="heading-row"><div class="heading-spacer"></div>'
        '<div class="column-heading">Masked (POS Tags)</div>'
        '<div class="column-heading">Unmasked (Latin Text)</div>'
        "</div>" + html + "</body></html>"
    )
    html_bytes = output_html.encode("utf-8")
    html_b64 = base64.b64encode(html_bytes).decode("utf-8")
    return IFrame(f"data:text/html;base64,{html_b64}", width="100%", height=800)


def top_n(result: dict, n: int, title: str) -> IFrame:
    """Extract the top N sentences (most similar = first in order_by=importance)."""
    rows = extract_sentence_rows(result["colourcoded_text"])[:n]
    return render_rows(rows, title, f"Top {n} sentences (highest lambdaG)")


def bottom_n(result: dict, n: int, title: str) -> IFrame:
    """Extract the bottom N sentences (least similar = last in order_by=importance)."""
    rows = extract_sentence_rows(result["colourcoded_text"])[-n:]
    rows = list(reversed(rows))  # most dissimilar first
    return render_rows(rows, title, f"Bottom {n} sentences (lowest lambdaG)")


# Strip pipes from unmasked text
unmasked_clean = strip_pipes(unmasked)

rng = np.random.default_rng(42)

# Run all 4 configurations and store results
results = {}
for label, q, k, ref, neg in [
    ("Metrical_Vergil",    to_sentence_lists(m_cento), to_sentence_lists(m_vergil),  to_sentence_lists(m_background), True),
    # ("Metrical_Ausonius",  to_sentence_lists(m_cento), to_sentence_lists(m_ausonius), to_sentence_lists(m_background), True),
    # ("Syntactic_Vergil",   to_sentence_lists(s_cento), to_sentence_lists(s_vergil),  to_sentence_lists(s_background), False),
    # ("Syntactic_Ausonius", to_sentence_lists(s_cento), to_sentence_lists(s_ausonius), to_sentence_lists(s_background), True),
]:
    print(f"Running {label}...", end=" ", flush=True)
    results[label] = run_lambdag(q, unmasked_clean, k, ref, neg, rng)
    print("done")


Running Metrical_Vergil... done


In [7]:
# Full raw lambdaG output — all cento sentences vs Vergil (metrical)
display(render_full(results["Metrical_Vergil"], "Cento vs Vergil (Georgics) — Metrical", len(m_cento)))

In [9]:

# Top 10 Vergil sentences (metrical) — should match the first 10 in the full visualization
display(bottom_n(results["Metrical_Vergil"], 10, "Top 10 Cento Sentences Most Like Vergil — Metrical"))


In [10]:
# Load Mosella in both representations
m_mosella = load_corpus(Path("metrical_model/ausonius"), "_sentences.metrical.txt")
m_mosella = TwoLevelCorpus(
    [s for s in m_mosella if s.metadata.get("work") == "mosella"],
    levels=["author", "work"],
)

s_mosella = load_corpus(Path("lexical_model/ausonius"), "_sentences.quesplit.masked.txt")
s_mosella = TwoLevelCorpus(
    [s for s in s_mosella if s.metadata.get("work") == "mosella"],
    levels=["author", "work"],
)

print(f"Mosella metrical: {len(m_mosella)} sentences, {sum(len(s.text.split()) for s in m_mosella)} tokens")
print(f"Mosella syntactic: {len(s_mosella)} sentences, {sum(len(s.text.split()) for s in s_mosella)} tokens")

Mosella metrical: 163 sentences, 3577 tokens
Mosella syntactic: 163 sentences, 3688 tokens


In [12]:
mos_metr = run_lambdag(to_sentence_lists(m_mosella), to_sentence_lists(unmasked_clean), to_sentence_lists(m_vergil), to_sentence_lists(m_background), True, rng)

AttributeError: 'str' object has no attribute 'text'